# Gemma 4: recall and error analysis

This notebook analyzes two saved Gemma runs on the same 559 curated Enron test emails. It reads local files only and makes no API calls. The `exclude` rule rejects third-party news; the `include` rule follows the original labels' broader definition.

**For recall on the original labels, use the include-news rule: 67/73 = 0.918.** The exclude-news rule alone reaches 19/73 = 0.260 on those labels, because 53 of its 54 false negatives are annotated news. On the intended non-news subset it reaches 19/20 = 0.950. The rule-assisted news audit is not a blinded gold standard.


In [ ]:
import csv
import math
from collections import Counter, defaultdict
from pathlib import Path

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data' / 'enron_labeled_curated_test.csv').exists()), None)
if REPO is None:
    raise FileNotFoundError('Start Jupyter within the GTPracticum repository')
DATA = REPO / 'data'
TEST = DATA / 'enron_labeled_curated_test.csv'
GEMMA = DATA / 'gemma4_exclude_test_predictions.csv'
JEV = DATA / 'jev_test_predictions.csv'

with TEST.open(newline='', encoding='utf-8-sig') as f:
    emails = list(csv.DictReader(f))
with GEMMA.open(newline='', encoding='utf-8') as f:
    saved = list(csv.DictReader(f))
rows = []
seen = set()
for p in saved:
    i = int(p['source_row'])
    if i in seen or i < 0 or i >= len(emails):
        raise ValueError(f'Duplicate or invalid source row: {i}')
    seen.add(i)
    truth = int(emails[i]['label'])
    if int(p['label']) != truth:
        raise ValueError(f'Label mismatch at row {i}')
    rows.append({**p, 'source_row': i, 'label': truth,
                 'prediction': int(p['prediction']), 'email': emails[i]['email']})
rows.sort(key=lambda r: r['source_row'])
print(f'Loaded {len(rows)}/{len(emails)} saved predictions')
print('Class balance:', Counter(r['label'] for r in rows))
print('Source types:', Counter(r['source_type'] for r in rows))
print('Complete test set:', len(rows) == len(emails))


## What precision and recall measure

Precision = TP / (TP + FP): among emails Gemma flags, how many have a positive curated label? Recall = TP / (TP + FN): among positive curated labels, how many does Gemma find? F1 is their harmonic mean. A news email that the curated dataset labels positive but this Gemma rule excludes will count as a false negative.


In [ ]:
def scores(items):
    n = len(items)
    tp = sum(r['label'] == 1 and r['prediction'] == 1 for r in items)
    tn = sum(r['label'] == 0 and r['prediction'] == 0 for r in items)
    fp = sum(r['label'] == 0 and r['prediction'] == 1 for r in items)
    fn = sum(r['label'] == 1 and r['prediction'] == 0 for r in items)
    precision = tp / (tp + fp) if tp + fp else float('nan')
    recall = tp / (tp + fn) if tp + fn else float('nan')
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else float('nan')
    return dict(n=n, TP=tp, TN=tn, FP=fp, FN=fn,
                precision=precision, recall=recall, f1=f1,
                accuracy=(tp + tn) / n if n else float('nan'))

overall = scores(rows)
for k, v in overall.items():
    print(f'{k:10} {v:.3f}' if isinstance(v, float) else f'{k:10} {v}')
print('\nConfusion matrix (rows: curated label; columns: Gemma prediction)')
print('                 predicted 0  predicted 1')
print(f"actual 0          {overall['TN']:>5}        {overall['FP']:>5}")
print(f"actual 1          {overall['FN']:>5}        {overall['TP']:>5}")


## Precision and recall by model-identified source type

Gemma assigns `source_type` alongside its label. This is useful for finding news-related disagreements, but it is a model output, not a verified source annotation. Counts for each subset use the same curated labels. Small subsets have noisy rates.


In [ ]:
groups = {name: [r for r in rows if r['source_type'] == name]
          for name in ('internal', 'news', 'other')}
print(f"{'source':<12} {'n':>5} {'TP':>5} {'FP':>5} {'FN':>5} {'precision':>11} {'recall':>9}")
for name, items in groups.items():
    s = scores(items) if items else None
    if s:
        print(f"{name:<12} {s['n']:>5} {s['TP']:>5} {s['FP']:>5} {s['FN']:>5} {s['precision']:>11.3f} {s['recall']:>9.3f}")
print('News rows with positive curated label:',
      sum(r['label'] == 1 for r in groups['news']))


## Check the separate news label

`enron_labeled_curated_test_with_news.csv` adds a binary publication label without copying Gemma predictions. It marks copied articles, newswires, market bulletins, and analyst reports as news, including forwarded copies. It was built from publication markers plus row-level review of ambiguous excerpts. Because some review candidates were selected after seeing Gemma's source types, agreement rates are descriptive and may be optimistic; the label is an audit aid rather than a blinded gold standard.


In [ ]:
NEWS_TEST = DATA / 'enron_labeled_curated_test_with_news.csv'
with NEWS_TEST.open(newline='', encoding='utf-8') as f:
    annotated = list(csv.DictReader(f))
if len(annotated) != len(emails) or any(a['email'] != e['email'] for a, e in zip(annotated, emails)):
    raise ValueError('News annotations do not match the test set')
for r in rows:
    a = annotated[r['source_row']]
    r['news_label'] = int(a['news_label'])
    r['news_label_basis'] = a['news_label_basis']
print('Annotated news:', sum(r['news_label'] for r in rows))
print('Gemma accounting positives among annotated news:',
      sum(r['prediction'] == 1 for r in rows if r['news_label'] == 1))
print('Gemma source-type check (news label, predicted news):')
print(Counter((r['news_label'], int(r['source_type'] == 'news')) for r in rows))
non_news = [r for r in rows if r['news_label'] == 0]
s = scores(non_news)
print(f"Accounting precision/recall on annotated non-news only: {s['precision']:.3f} / {s['recall']:.3f} (n={s['n']})")
disagreements = [r for r in rows if r['news_label'] != (r['source_type'] == 'news')]
for r in disagreements[:20]:
    print(f"row {r['source_row']}: news_label={r['news_label']} Gemma_source={r['source_type']} | {r['email'][:180].replace(chr(10), ' ')}")


## Recall under the intended label policy

For this classifier, a positive requires both the original accounting label and a non-news source. The cell below computes this policy-aligned label without altering the source CSV, then prints every remaining false negative. There are only 20 positive examples under this definition, so the 0.950 recall estimate is sensitive to one case.


In [ ]:
policy_rows = [{**r, 'label': int(r['label'] == 1 and r['news_label'] == 0)} for r in rows]
policy = scores(policy_rows)
print('Policy-aligned recall:', f"{policy['TP']}/{policy['TP'] + policy['FN']} = {policy['recall']:.3f}")
print('Policy-aligned precision:', f"{policy['TP']}/{policy['TP'] + policy['FP']} = {policy['precision']:.3f}")
print('Remaining false negatives:')
for r in policy_rows:
    if r['label'] == 1 and r['prediction'] == 0:
        print('row', r['source_row'], 'reason:', r['reason'])
        print(r['email'][:1200])


## Recall-first comparison: include versus exclude news

These runs use the same Gemma model and test emails but different news rules. The `include` run matches the original benchmark's broader positive definition more closely. Compare recall on the original labels first, then inspect where extra retrieved positives and false positives come from. Neither run was fine-tuned on the test labels.


In [ ]:
INCLUDE = DATA / 'gemma4_include_test_predictions.csv'
if INCLUDE.exists():
    with INCLUDE.open(newline='', encoding='utf-8') as f:
        include_saved = list(csv.DictReader(f))
    include_by_row = {int(r['source_row']): r for r in include_saved}
    if len(include_by_row) != len(rows):
        print(f'Include run is incomplete: {len(include_by_row)}/{len(rows)} rows')
    else:
        include_rows = [{**r, 'prediction': int(include_by_row[r['source_row']]['prediction']),
                         'reason': include_by_row[r['source_row']]['reason']} for r in rows]
        print(f"{'rule':<16} {'recall':>8} {'precision':>10} {'TP':>5} {'FN':>5} {'FP':>5}")
        for name, subset in [('exclude news', rows), ('include news', include_rows)]:
            s = scores(subset)
            print(f"{name:<16} {s['recall']:>8.3f} {s['precision']:>10.3f} {s['TP']:>5} {s['FN']:>5} {s['FP']:>5}")
        for source in (0, 1):
            print('Annotated news =', source)
            for name, subset in [('exclude', rows), ('include', include_rows)]:
                s = scores([r for r in subset if r['news_label'] == source])
                print(name, 'recall=', round(s['recall'], 3), 'positives=', s['TP'] + s['FN'], 'FP=', s['FP'])
else:
    print('Include-news run is not available yet.')


## Precision–recall figure

Gemma emits one discrete label per email. This figure compares the single operating point across source types, with Wilson 95% intervals for each proportion. It is **not** a threshold-sweep precision–recall curve because Gemma did not return calibrated scores. Run `%pip install matplotlib` in the notebook environment if the package is missing.


In [ ]:
import matplotlib.pyplot as plt

def wilson(successes, trials, z=1.96):
    if not trials:
        return (float('nan'), float('nan'))
    p = successes / trials
    denom = 1 + z*z/trials
    center = (p + z*z/(2*trials)) / denom
    half = z * math.sqrt(p*(1-p)/trials + z*z/(4*trials*trials)) / denom
    return center-half, center+half

named = [('all', rows)] + [(name, items) for name, items in groups.items() if items]
fig, ax = plt.subplots(figsize=(7, 5))
for name, items in named:
    s = scores(items)
    if math.isnan(s['precision']) or math.isnan(s['recall']):
        continue
    p_lo, p_hi = wilson(s['TP'], s['TP'] + s['FP'])
    r_lo, r_hi = wilson(s['TP'], s['TP'] + s['FN'])
    ax.errorbar(s['recall'], s['precision'],
                xerr=[[s['recall']-r_lo], [r_hi-s['recall']]],
                yerr=[[s['precision']-p_lo], [p_hi-s['precision']]],
                marker='o', capsize=3, label=f"{name} (n={s['n']})")
ax.set(xlim=(0, 1.02), ylim=(0, 1.02), xlabel='Recall', ylabel='Precision',
       title='Gemma 4 classification points (95% Wilson intervals)')
ax.grid(alpha=.25)
ax.legend()
plt.show()


## Review false positives and false negatives

The excerpts below show the evidence available to the model. Read the full email and curated labeling notes before changing a label: a short excerpt or model-generated reason may omit decisive context. In particular, separate genuine internal accounting concerns from third-party reporting.


In [ ]:
def show_errors(kind, limit=20, chars=650):
    truth, pred = {'FP': (0, 1), 'FN': (1, 0)}[kind]
    errors = [r for r in rows if r['label'] == truth and r['prediction'] == pred]
    print(f'{kind}: {len(errors)} total; showing {min(limit, len(errors))}')
    for r in errors[:limit]:
        print(f"\nrow {r['source_row']} | source={r['source_type']} | truncated={r['truncated']}")
        print('Gemma reason:', r['reason'])
        print('Email:', r['email'][:chars].replace('\n', ' '))
    return errors

false_positives = show_errors('FP')


In [ ]:
false_negatives = show_errors('FN')


## Context: Jev on the same test set

This comparison displays saved Jev metrics if its CSV is present. Jev's positive rule includes relevant news; Gemma's default excludes it. Score differences therefore reflect both model behavior and label policy. Gemma's source types and explanations are generated text; Jev supplies a numeric probability.


In [ ]:
if JEV.exists():
    with JEV.open(newline='', encoding='utf-8') as f:
        jev_saved = list(csv.DictReader(f))
    jev_rows = [{'label': int(r['label']), 'prediction': int(r['prediction'])}
                for r in jev_saved]
    for name, s in [('Gemma (news excluded)', overall), ('Jev (news included)', scores(jev_rows))]:
        print(f"{name:<23} n={s['n']} precision={s['precision']:.3f} recall={s['recall']:.3f} F1={s['f1']:.3f} FP={s['FP']} FN={s['FN']}")
else:
    print('Jev results CSV not found; no comparison shown.')
